## Modèle simple pour développer l'intuition

In [9]:
# import librairies

import pyomo.environ as pyo

In [10]:
# create model

m = pyo.ConcreteModel()

### Produits comptables

In [11]:
# MAD facturée aux partenaires (unités = M€) (NB : >=0) pour les projets de recherche en propre
m.mad = pyo.Var(bounds=(5.0, 10.0))  

# contribution partenaires en cash (et assimilé) (unités = M€) pour les projets de recherche en propre
m.cash = pyo.Var(bounds=(5.0, 8.0)) 

# autres produits, hors projets de recherche en propre : 
# prestations, AAPs type HE, CNES, AID..., autres subventions
m.autres = pyo.Var(bounds=(3.0, 8.0))  

# France 2030 socle, cappé à 11M€ par Convention 2026.
m.f2030 = pyo.Var(bounds=(0.0, 11.0))  

# Ligne totale
m.produits = pyo.Expression(
    expr=m.mad + m.cash + m.autres + m.f2030
)

### Charges comptables

In [12]:
# masse salariale IRT, chargée, projetée (=liquidée) sur projets. 
# Vue comptable, pas analytique : hors "indirects"
m.w_proj = pyo.Var(bounds=(7.0, 10.0))   

# masse salariale IRT, chargée, projetable sur projets uniquement via imputation "environnement"
# sur coûts projets complets (RF ANR, RGEC)
# typiquement : fonctions support, intercontrats, R&D innovation
m.w_ind = pyo.Var(bounds=(7.0, 10.0))

# autres coûts fixes de structure (locaux, électricité, etc.)
m.structure = pyo.Param(initialize=1.0)  # A REVOIR AVEC UNE VRAIE ESTIMATION

# MAD facturée par les partenaires
# => il s'agit (au signe près) de m.mad qui apparaît en produits plus haut (double facturation)

# achats et sous-traitance sur projets
# distinguo rendu nécessaires par règle des 50% max sous-traitance sur projets
m.x_achats = pyo.Var(bounds=(5.0, 7.0))   # unités en M€. VALEUR ?
m.x_st = pyo.Var(bounds=(5.0, 7.0))   # untiés en M€. VALEUR ?

# amortissements (charge calculée)
m.amortissements = pyo.Param(initialize=1.0)   # VRAIE VALEUR ?

# frais financiers, impôts, provisions
m.frais_fi = pyo.Param(initialize=0.0)   # VRAIE VALEUR ?
m.provisions = pyo.Param(initialize=0.0)   # VRAIE VALEUR ?

# Ligne totale
m.charges = pyo.Expression(
    expr=m.w_proj + m.w_ind + m.structure + m.mad + m.x_achats + m.x_st + m.amortissements + m.frais_fi + m.provisions
)

### Régles

Conventions

In [13]:
# 1. Dotation France 2030 Socle cappée à 11 M€ : voir plus haut
# m.f2030 = pyo.Var(bounds=(0.0, 11.0))  

In [14]:
# 2. Dotation France 2030 inférieure au tiers des dépenses (comptables) de l'Institut
m.c_f2030 = pyo.Constraint(
    expr=m.f2030 <= (1/3) * m.charges
)

In [15]:
# 3. Cofinancements privés projets de recherche en propre >= France 2030 socle
m.c_cofinancements = pyo.Constraint(
    expr=m.mad + m.cash >= m.f2030
)

### RGEC, RF ANR, intensités financement, etc.

A revoir avec la finance a minima

In [16]:
# 4. Intensité de financement sur projets de recherche en propre
# ici : supposée égale à 50% (soit RI, Grande Entreprise, hors collab et grande diffusion)
# NB : on peut introduire un mix de projets (RF,RI, DE, Faisabilité) avec intensités différentes.

m.intensite = pyo.Param(initialize=0.50)  # 50% ici, customisable

# calcul des coûts environnement - A REVOIR AVEC LA FINANCE, ici RF ANR :
# soit 68% coûts personnels admissibles + 7% autres coûts, 
# mais cappé par MO non projetée + coûts fixes structures
m.couts_environnement = pyo.Expression(
    expr=0.68 * (m.w_proj+m.mad) + 0.07*(m.x_achats + m.x_st + m.amortissements)
)
m.c_couts_environnement_cappes = pyo.Constraint(
    expr=m.couts_environnement <= m.w_ind + m.structure
)

# assiette coûts complets (RGEC, RF ANR 3/7/2025)
m.couts_complets = pyo.Expression(
    expr=m.w_proj + m.mad + m.x_achats + m.x_st + m.amortissements + m.couts_environnement
)

# contrainte proprement dite
m.c_intensite = pyo.Constraint(
    expr=m.f2030 <= m.intensite * m.couts_complets
)

### Sous-traitance

In [17]:
# 5. part maximale de sous-traitance

m.c_st = pyo.Constraint(
    expr=m.x_st <= 0.50*m.couts_complets
)

### Résultat comptable

In [18]:
m.resultat = pyo.Expression(
    expr=m.produits - m.charges
)

### Objectif

In [19]:
# on introduit une variable slack >= 0, la contrainte resultat + slack >= 0, et on va minimiser slack
#
# si la valeur slack=0 est atteinte, cela veut dire qu'il existe un m.resultat >= 0. 
# L'Institut est viable sous les contraintes et intervalles de variables donnés
#
# si la valeur optimale est slack>0, cela veut dire que le meilleur resultat est négatif. 
# slack mesure l'écart comptable à l'équilibre.

m.slack_variable = pyo.Var(within=pyo.NonNegativeReals)

m.c_resultat = pyo.Constraint(
    expr=m.resultat + m.slack_variable >= 0.0
)

m.objectif = pyo.Objective(expr=m.slack_variable, sense=pyo.minimize)

### Vérifications diverses

In [20]:
# 1. Check structure
m.display()

# 2. Write LP file (best consistency check)
try:
    m.write('model_check.lp')
    print("Model is syntactically valid")
except Exception as e:
    print(f"Model error: {e}")

Model unknown

  Variables:
    mad : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :   5.0 :  None :  10.0 : False :  True :  Reals
    cash : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :   5.0 :  None :   8.0 : False :  True :  Reals
    autres : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :   3.0 :  None :   8.0 : False :  True :  Reals
    f2030 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :   0.0 :  None :  11.0 : False :  True :  Reals
    w_proj : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :   7.0 :  None :  10.0 : False :  True :  Reals
    w_ind : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :   7.0 :  None :  10.0 : False :  True :  Reals
    x_achats : Size=1, Index=None
        K

### Résolution

In [21]:
# Solve the model
solver = pyo.SolverFactory('appsi_highs')
results = solver.solve(m)

# Show solver report
results.write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Lower bound: 0.28666666666666707
  Upper bound: 0.28666666666666707
  Number of objectives: 1
  Number of constraints: 0
  Number of variables: 0
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Termination message: TerminationCondition.optimal
# ----------------------------------------------------------
#   Solution Information
# ----------------------------------------------------------
Solution: 
- number of solutions: 0
  number of solutions displayed: 0


### Résultats

In [22]:
print(f"Grandeurs macros (M€):")
print(f"Produits : {pyo.value(m.produits):.2f}, charges : {pyo.value(m.charges):.2f}")
print(f"Résultat : {pyo.value(m.resultat):.2f}")
print(f"\nMinimized slack (amélioration nécessaire pour équilibre): {pyo.value(m.slack_variable):.2f}")

print("\nDétails Solution:")

print("\n1. Valeurs variables")
for v in m.component_data_objects(pyo.Var):
    print(f"{v.name} = {pyo.value(v):.2f}")
    
print("\n2. Valeurs expressions intermédiaires")
for expr in m.component_data_objects(pyo.Expression):
    print(f"{expr.name} = {pyo.value(expr):.2f}")
    
print("\n3. Valeurs paramètres")
for para in m.component_data_objects(pyo.Param):
    print(f"{para.name} = {pyo.value(para):.2f}")

Grandeurs macros (M€):
Produits : 31.64, charges : 31.93
Résultat : -0.29

Minimized slack (amélioration nécessaire pour équilibre): 0.29

Détails Solution:

1. Valeurs variables
mad = 5.00
cash = 8.00
autres = 8.00
f2030 = 10.64
w_proj = 7.00
w_ind = 7.93
x_achats = 5.00
x_st = 5.00
slack_variable = 0.29

2. Valeurs expressions intermédiaires
produits = 31.64
charges = 31.93
couts_environnement = 8.93
couts_complets = 31.93
resultat = -0.29

3. Valeurs paramètres
structure = 1.00
amortissements = 1.00
frais_fi = 0.00
provisions = 0.00
intensite = 0.50
